# 02 The chain

This notebook turns the graph into the random surfer's Markov chain, computes its
stationary distribution π for five values of the damping factor α, and checks the
result.

In [6]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from src.load import load_graph
from src.chain import build_chain, step, stationary

A, node_ids, info = load_graph()
out_deg = np.asarray(A.sum(axis=1)).ravel().astype(int)
in_deg = np.asarray(A.sum(axis=0)).ravel().astype(int)

The random surfer moves as follows. With probability α it follows a link chosen
uniformly from the current page, and with probability 1 − α it jumps to a page chosen uniformly from the whole graph. On a dangling page it always jumps.

If x is the distribution of the surfer's position, then

$$x_{k+1} = \alpha\Big(P^{\top}x_k + \Big(\sum_{i\ \text{dangling}} x_k(i)\Big)u\Big) + (1-\alpha)\,u$$

where P is the link matrix with each row divided by its out-degree and u is the
uniform distribution.

In [7]:
PT, dangling = build_chain(A)
ALPHAS = [0.50, 0.70, 0.85, 0.95, 0.99]

rows = []
for alpha in ALPHAS:
    pi, steps = stationary(PT, dangling, alpha)
    residual = np.abs(step(pi, PT, dangling, alpha) - pi).sum()
    np.save(f"../data/pi_alpha_{alpha:.2f}.npy", pi)
    rows.append({"alpha": alpha, "steps": steps, "residual": residual})

pd.DataFrame(rows)

,alpha,steps,residual
0,0.50,50,1.858284e-17
1,0.70,97,2.796119e-17
2,0.85,213,2.195660e-16
3,0.95,674,3.598931e-16
4,0.99,3437,1.998726e-15


## Reference distributions

For each α the chain is run from the uniform distribution until α^k is below
10⁻¹⁵, and the result is saved as the reference π. The residual is the L1 change
from applying one more step. 

In [8]:
pi = np.load("../data/pi_alpha_0.85.npy")
top = np.argsort(pi)[::-1][:10]
pd.DataFrame({
    "SNAP id": node_ids[top],
    "PageRank": pi[top],
    "in-links": in_deg[top],
    "out-links": out_deg[top],
})

,SNAP id,PageRank,in-links,out-links
0,89073,0.011303,15277,64
1,226411,0.009268,38606,20
2,241454,0.008297,19377,64
3,262860,0.003023,1802,1
4,134832,0.003001,10336,7
5,234704,0.002572,21920,23
6,136821,0.002454,4906,14
7,68889,0.002431,4907,12
8,105607,0.002391,19457,35
9,69358,0.002364,13936,7


As the foundational understanding of PageRank suggests, rank is not the same as in-link count. The page with the most in-links (38,606) is ranked second. We know that within this system, a link carries more weight when it comes from a page that is itself highly ranked.

The top-ranked page holds 1.13% of the stationary mass and has 15,277 in-links.